In [ ]:
import pymupdf  # PyMuPDF
import re
from pathlib import Path

In [ ]:
# === KONFIGURACJA ===
PDF_PATH = Path("regulamin studiów AHE.pdf")
OUTPUT_PATH = Path("regulamin_oczyszczony.txt")

START_HEADING = "REGULAMIN STUDIÓW"


def extract_pdf_text(pdf_path: Path) -> str:
    """Wyciąga tekst ze wszystkich stron PDF-a."""
    text_parts = []

    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            text = page.get_text("text", sort=True)
            text_parts.append(
                text
            )

    return "\n".join(text_parts)


def remove_page_markers(text: str) -> str:
    """Usuwa nagłówki techniczne i numery stron."""
    # Nasze znaczniki techniczne stron
    text = re.sub(r"(?m)^--- STRONA \d+ ---\s*$", "", text)

    # Numery stron w postaci: - 7 -, – 7 – itp.
    text = re.sub(
        r"(?m)^\s*[-–—]\s*\d+\s*[-–—]\s*$",
        "",
        text
    )

    # Numery stron będące samodzielnym wierszem
    # Usuwaj tylko wtedy, gdy występują jako osobny wiersz.
    return text


def remove_amendment(text: str) -> str:
    """Zostawia tekst od nagłówka właściwego regulaminu."""
    match = re.search(
        re.escape(START_HEADING),
        text,
        flags=re.IGNORECASE
    )

    if not match:
        raise ValueError(
            f"Nie znaleziono nagłówka '{START_HEADING}'. "
            "Sprawdź, czy PDF zawiera tekst, a nie wyłącznie skany."
        )

    return text[match.start():]


def fix_hyphenated_words(text: str) -> str:
    """
    Łączy wyrazy przeniesione do kolejnego wiersza:
    stu-
    dent -> student

    Zachowuje zwykłe łączniki wewnątrz wyrazów.
    """
    return re.sub(
        r"(?<=[A-Za-zĄĆĘŁŃÓŚŹŻąćęłńóśźż])-\s*\n\s*"
        r"(?=[A-Za-zĄĆĘŁŃÓŚŹŻąćęłńóśźż])",
        "",
        text
    )


def normalize_lines(text: str) -> str:
    """Łączy przełamane wiersze, zachowując podział na akapity."""
    text = fix_hyphenated_words(text)

    text = text.replace("\r\n", "\n").replace("\r", "\n")

    text = re.sub(r"\n[ \t]*\n+", "\n\n", text)

    lines = text.split("\n")
    result = []

    for line in lines:
        line = line.strip()

        if not line:
            if result and result[-1] != "":
                result.append("")
            continue

        if not result or result[-1] == "":
            result.append(line)
            continue

        previous = result[-1]

        starts_new_section = bool(
            re.match(
                r"^(§\s*\d+|"
                r"\d+\.\s|"
                r"\d+\)\s|"
                r"[a-z]\)\s|"
                r"USTAWA|REGULAMIN STUDIÓW|"
                r"Załącznik\b)",
                line,
                flags=re.IGNORECASE
            )
        )

        # Zachowaj osobno nagłówki paragrafów i punktów.
        if starts_new_section:
            result.append("")
            result.append(line)
        else:
            # Dodaj spację między wierszami tekstu.
            result[-1] = previous + " " + line

    text = "\n".join(result)

    # Usuń spacje przed interpunkcją.
    text = re.sub(r"[ \t]+([,.;:!?])", r"\1", text)

    # Uporządkuj odstępy
    text = re.sub(r"[ \t]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

def remove_section_numbers(text: str) -> str:
    """Usuwa numery ustępów na początku wiersza, np. 1., 2., 3."""
    return re.sub(
        r"(?m)^[ \t]*\d+\.[ \t]+(?=\S)",
        "",
        text
    )

def format_criteria_table(text: str) -> str:
    """
    Miejsce na ujednolicenie tabeli z § 18 ust. 10.
    Najpierw pozostawiamy oryginalne wartości,
    a następnie można je przepisać do czytelnych wierszy.
    """
    return text


def format_rounding_table(text: str) -> str:
    """
    Miejsce na ujednolicenie tabeli z § 32 ust. 6.
    Sposób formatowania zależy od rzeczywistego układu PDF-a.
    """
    return text


if not PDF_PATH.exists():
    raise FileNotFoundError(
        f"Nie znaleziono pliku: {PDF_PATH.resolve()}"
    )

# 1. Ekstrakcja oryginalnego tekstu
raw_text = extract_pdf_text(PDF_PATH)

# 2. Właściwy regulamin od strony 3
regulation_text = remove_amendment(raw_text)

# 3. Czyszczenie
cleaned_text = remove_page_markers(regulation_text)
cleaned_text = normalize_lines(cleaned_text)



# 4. Formatowanie tabel — do dopasowania do treści PDF-a
cleaned_text = format_criteria_table(cleaned_text)
cleaned_text = format_rounding_table(cleaned_text)

raw_section = re.search(
    r"§\s*18\b.*?(?=§\s*19\b)",
    regulation_text,
    flags=re.DOTALL
)
clean_section = re.search(
    r"§\s*18\b.*?(?=§\s*19\b)",
    cleaned_text,
    flags=re.DOTALL
)

print("\n" + "=" * 70)
print("FRAGMENT PRZED CZYSZCZENIEM")
print("=" * 70)
print(raw_section.group(0)[:3000] if raw_section else
        "Nie znaleziono § 18 w wyodrębnionym tekście.")

print("\n" + "=" * 70)
print("TEN SAM FRAGMENT PO CZYSZCZENIU")
print("=" * 70)
print(clean_section.group(0)[:3000] if clean_section else
        "Nie znaleziono § 18 w oczyszczonym tekście.")





FRAGMENT PRZED CZYSZCZENIEM
§ 18.
1.  Efekty uczenia się uzyskane przez studenta w toku studiów podlegają weryfikacji.
2.  Nieprzystąpienie przez studenta do weryfikacji efektów uczenia  się w wyznaczonych
    terminach uniemożliwia uzyskanie zaliczenia tych efektów w ramach zajęć, semestru lub
    roku studiów.
3.  Weryfikacja efektów uczenia się może być dokonywana w czasie:
         a.  trwania semestru;


                                                                  - 8 -
        b. w czasie sesji egzaminacyjnej, która może być:
         – sesją zwykłą,
         – sesją poprawkową.
4.  Studentowi przysługują dwa terminy zaliczeniowe lub egzaminacyjne – pierwszy termin
    oraz termin poprawkowy. W przypadku egzaminów pierwszy termin wyznaczony jest w
     sesji zwykłej, termin poprawkowy – w sesji poprawkowej.
5.  Egzaminy mogą  odbywać  się  tylko w  czasie  sesji  egzaminacyjnej  (zwykłej  lub
    poprawkowej) wyznaczonej przez harmonogram roku akademickiego.
6.  Terminy ora

In [26]:

import re
import json
import statistics
from pathlib import Path

import tiktoken


# =========================
# KONFIGURACJA
# =========================

INPUT_PATH = Path("regulamin_oczyszczony.txt")
OUTPUT_PATH = Path("fragmenty_regulaminu.json")

# Docelowy limit dla jednego fragmentu.
# To limit miękki: długie ustępy dzielimy na mniejsze części.
MAX_TOKENS = 500

# Nakładanie fragmentów przy dzieleniu długiego ustępu.
OVERLAP_TOKENS = 50

# Tokenizer do pomiaru tekstu.
# cl100k_base jest często używanym tokenizerem w zastosowaniach RAG.
ENCODING = tiktoken.get_encoding("cl100k_base")


# =========================
# TOKENIZACJA
# =========================

def count_tokens(text: str) -> int:
    return len(ENCODING.encode(text))


def split_by_tokens(text: str, max_tokens: int, overlap_tokens: int):
    """
    Dzieli długi tekst na fragmenty nieprzekraczające limitu tokenów.
    Stara się kończyć fragmenty na granicach zdań.
    """
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())

    chunks = []
    current = []

    for sentence in sentences:
        candidate = " ".join(current + [sentence])

        if count_tokens(candidate) <= max_tokens:
            current.append(sentence)
            continue

        if current:
            chunks.append(" ".join(current))

        # Jeśli pojedyncze zdanie jest za długie,
        # dzielimy je na kawałki tokenów.
        if count_tokens(sentence) > max_tokens:
            token_ids = ENCODING.encode(sentence)
            start = 0

            while start < len(token_ids):
                end = min(start + max_tokens, len(token_ids))
                piece = ENCODING.decode(token_ids[start:end])
                chunks.append(piece)

                if end == len(token_ids):
                    break

                start = max(start + 1, end - overlap_tokens)

            current = []
        else:
            # Opcjonalny overlap z poprzedniego fragmentu.
            previous = chunks[-1] if chunks else ""
            previous_sentences = re.split(
                r"(?<=[.!?])\s+", previous
            )

            overlap = []
            for old_sentence in reversed(previous_sentences):
                candidate_overlap = [old_sentence] + overlap

                if count_tokens(" ".join(candidate_overlap)) > overlap_tokens:
                    break

                overlap = candidate_overlap

            current = overlap + [sentence]

    if current:
        chunks.append(" ".join(current))

    return [chunk.strip() for chunk in chunks if chunk.strip()]


# =========================
# STRUKTURA REGULAMINU
# =========================

def split_paragraphs(text: str):
    """
    Rozpoznaje nagłówki paragrafów, np. § 1, § 2, § 30.
    Numer paragrafu jest pobierany z tekstu, a nie wyliczany.
    """
    matches = list(re.finditer(r"(?m)^\s*§\s*(\d+[a-zA-Z]?)\b", text))

    paragraphs = []

    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)

        paragraphs.append({
            "paragraf": match.group(1),
            "text": text[start:end].strip()
        })

    return paragraphs


def split_into_sections(paragraph_text: str, paragraph_number: str):
    """
    Dzieli paragraf na ustępy, zachowując numerację z dokumentu.
    Nie zakłada, że pierwszy ustęp musi mieć numer 1.
    """
    # Usuwamy nagłówek paragrafu, ale zachowujemy jego treść.
    body = re.sub(
        r"^\s*§\s*\d+[a-zA-Z]?\b[^\n]*",
        "",
        paragraph_text,
        count=1
    ).strip()

    # Numer ustępu musi występować na początku wiersza.
    # Dzięki temu liczby wewnątrz zdań nie są uznawane za ustępy.
    matches = list(re.finditer(r"(?m)^\s*(\d+)\.\s+", body))

    if not matches:
        return [{
            "paragraf": paragraph_number,
            "ustep": 0,
            "text": paragraph_text.strip()
        }]

    sections = []

    # Treść przed pierwszym numerowanym ustępem, jeśli istnieje.
    if body[:matches[0].start()].strip():
        sections.append({
            "paragraf": paragraph_number,
            "ustep": None,
            "text": body[:matches[0].start()].strip()
        })

    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(body)

        section_text = body[start:end].strip()

        # Usunięcie numeru ustępu z początku tekstu, np. "1. "
        section_text = re.sub(
            r"^\s*\d+\.\s+",
            "",
            section_text,
            count=1
        )

        sections.append({
            "paragraf": paragraph_number,
            "ustep": int(match.group(1)) or 0,
            "text": section_text
        })

    return sections


def build_chunks(text: str):
    paragraphs = split_paragraphs(text)
    result = []

    for paragraph in paragraphs:
        sections = split_into_sections(
            paragraph["text"],
            paragraph["paragraf"]
        )

        for section in sections:
            
        
            content = section["text"]
            token_count = count_tokens(content)

            if token_count <= MAX_TOKENS:
                result.append({
                    **section,
                    "fragment": 1,
                    "text": content,
                    "tokens": token_count
                })
                continue
        
            # Długi ustęp dzielimy dalej, zachowując metadane.
            smaller_chunks = split_by_tokens(
                content,
                MAX_TOKENS,
                OVERLAP_TOKENS
            )

            for index, chunk in enumerate(smaller_chunks, start=1):
                result.append({
                    "paragraf": section["paragraf"],
                    "ustep": section["ustep"] or 0,
                    "fragment": index,
                    "text": chunk,
                    "tokens": count_tokens(chunk)
                })

    return result


# =========================
# ZAPIS I STATYSTYKI
# =========================


text = cleaned_text

chunks = build_chunks(text)

with OUTPUT_PATH.open("w", encoding="utf-8") as file:
    json.dump(
        chunks,
        file,
        ensure_ascii=False,
        indent=2
    )

token_counts = [chunk["tokens"] for chunk in chunks]

print(f"Liczba fragmentów: {len(chunks)}")

if token_counts:
    print(
        "Typowa długość (mediana): "
        f"{statistics.median(token_counts):.0f} tokenów"
    )
    print(f"Średnia długość: {statistics.mean(token_counts):.1f} tokenów")
    print(f"Największa długość: {max(token_counts)} tokenów")
    print(f"Najmniejsza długość: {min(token_counts)} tokenów")

print(f"\nZapisano fragmenty do: {OUTPUT_PATH.resolve()}")

print("\n=== PRZYKŁADOWE FRAGMENTY ===")

for chunk in chunks[:15]:
    print("\n" + "-" * 70)
    print(
        f"§ {chunk['paragraf']}, "
        f"ustęp: {chunk['ustep']}, "
        f"fragment: {chunk['fragment']}, "
        f"tokeny: {chunk['tokens']}"
    )
    print(chunk["text"][:700])



Liczba fragmentów: 172
Typowa długość (mediana): 64 tokenów
Średnia długość: 79.9 tokenów
Największa długość: 487 tokenów
Najmniejsza długość: 6 tokenów

Zapisano fragmenty do: C:\Studia\Przetwarzanie tekstu\fragmenty_regulaminu.json

=== PRZYKŁADOWE FRAGMENTY ===

----------------------------------------------------------------------
§ 1, ustęp: 0, fragment: 1, tokeny: 91
§ 1. Regulamin stosuje się do wszystkich kierunków studiów, poziomów i profili kształcenia oraz form studiów prowadzonych przez Akademię Humanistyczno-Ekonomiczną w Łodzi (dalej: AHE w Łodzi lub Uczelnia) i dotyczy wszystkich studentów Uczelni.

----------------------------------------------------------------------
§ 2, ustęp: 1, fragment: 1, tokeny: 26
Akademia Humanistyczno-Ekonomiczna w Łodzi jest uczelnią niepubliczną.

----------------------------------------------------------------------
§ 2, ustęp: 2, fragment: 1, tokeny: 45
Po ukończeniu studiów absolwent otrzymuje tytuł zawodowy kierunku studiów, do którego 

In [27]:
import json
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

JSONL_PATH= Path("fragmenty_regulaminu.json")
INDEX_PATH = "regulamin.faiss"

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


def load_chunks(path: Path) -> list[dict]:
    with open(path, "r", encoding="utf-8") as file:
        try:
            chunks = json.load(file)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Błąd w pliku JSON: {exc}") from exc

    if not isinstance(chunks, list):
        raise ValueError("Plik JSON powinien zawierać tablicę fragmentów.")

    chunks = [
        chunk for chunk in chunks
        if isinstance(chunk, dict) and chunk.get("text")
    ]

    if not chunks:
        raise ValueError("Plik JSON nie zawiera żadnych fragmentów.")

    return chunks


# 1. Wczytanie dokumentów
chunks = load_chunks(JSONL_PATH)

# 2. Załadowanie modelu embeddingowego na CPU
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device="cpu",
)

# 3. Przygotowanie tekstów do indeksowania
texts = [
    (
        f"Regulamin studiów, paragraf {chunk['paragraf']}, ustęp {chunk['ustep']}. "
    ) + chunk["text"]
    for chunk in chunks
]

# 4. Generowanie wektorów
embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
)

embeddings = np.asarray(embeddings, dtype=np.float32)

# 5. Utworzenie indeksu FAISS
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

# 6. Zapis indeksu
faiss.write_index(index, INDEX_PATH)


print(f"Zaindeksowano {index.ntotal} chunków.")
print(f"Wymiar wektora: {dimension}")
print(f"Indeks zapisano w: {INDEX_PATH}")

Batches: 100%|██████████| 6/6 [00:03<00:00,  1.81it/s]

Zaindeksowano 172 chunków.
Wymiar wektora: 384
Indeks zapisano w: regulamin.faiss


In [28]:
texts

['Regulamin studiów, paragraf 1, ustęp 0. § 1. Regulamin stosuje się do wszystkich kierunków studiów, poziomów i profili kształcenia oraz form studiów prowadzonych przez Akademię Humanistyczno-Ekonomiczną w Łodzi (dalej: AHE w Łodzi lub Uczelnia) i dotyczy wszystkich studentów Uczelni.',
 'Regulamin studiów, paragraf 2, ustęp 1. Akademia Humanistyczno-Ekonomiczna w Łodzi jest uczelnią niepubliczną.',
 'Regulamin studiów, paragraf 2, ustęp 2. Po ukończeniu studiów absolwent otrzymuje tytuł zawodowy kierunku studiów, do którego nadawania uprawniona jest Uczelnia.',
 'Regulamin studiów, paragraf 2, ustęp 3. AHE w Łodzi prowadzi kształcenie w formie studiów stacjonarnych i niestacjonarnych.',
 'Regulamin studiów, paragraf 2, ustęp 4. Zajęcia dydaktyczne mogą być prowadzone z wykorzystaniem metod i technik kształcenia na odległość.',
 'Regulamin studiów, paragraf 2, ustęp 5. Przyjęcie na studia odbywa się w trybie określonym przez Senat Uczelni.',
 'Regulamin studiów, paragraf 3, ustęp 1. P